# Urban Step 3: Train & Fine-Tune Lightweight Object Detector Model

This notebook loads labeled YOLO format bounding box annotations (`.txt` files generated by `label_bbox_gui.py`), fine-tunes a lightweight detector for the **6 Urban Classes** (`green_light`, `red_light`, `turn_left_sign`, `turn_right_sign`, `stop_sign`, `crosswalk`), and exports BOTH PyTorch (`urban_detector.pth`) and ONNX (`urban_detector.onnx`, Opset 11) for Jetson Nano.

### 1. Setup Environment & Detection Dataset Loader

In [ ]:
import os
import sys
import glob
import time
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.utils.data as data
import torchvision.models as models
from pathlib import Path

# Add parent directory to sys.path
parent_dir = Path.cwd().parent.parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

try:
    from jetracer.urban.config import DETECTION_CLASSES
    from jetracer.utils import bgr8_to_jpeg
except ImportError:
    from urban.config import DETECTION_CLASSES
    from utils import bgr8_to_jpeg

# Lightweight MobileNetV2 Detection Head suitable for Jetson Nano
class MobileNetV2Detector(nn.Module):
    def __init__(self, num_classes=len(DETECTION_CLASSES), pretrained=True):
        super(MobileNetV2Detector, self).__init__()
        mobilenet = models.mobilenet_v2(pretrained=pretrained)
        self.features = mobilenet.features  # Output shape: (B, 1280, 7, 7)
        
        # Predict 4 bounding box coordinates + num_classes confidence scores
        self.bbox_head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(1280, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 4)
        )
        self.cls_head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(1280, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        feat = self.features(x)
        bboxes = torch.sigmoid(self.bbox_head(feat))  # Normalized [0, 1]
        classes = self.cls_head(feat)
        return bboxes, classes

class UrbanBBoxDataset(data.Dataset):
    def __init__(self, root_dirs):
        self.samples = []
        for d in root_dirs:
            txt_files = glob.glob(os.path.join(d, "*.txt"))
            for tf in txt_files:
                img_p = tf.replace('.txt', '.jpg')
                if os.path.exists(img_p):
                    try:
                        with open(tf, 'r') as f:
                            lines = f.readlines()
                        for line in lines:
                            parts = line.strip().split()
                            if len(parts) >= 5:
                                cls_id = int(parts[0])
                                cx, cy, bw, bh = map(float, parts[1:5])
                                self.samples.append({
                                    'image_path': img_p,
                                    'class_id': cls_id,
                                    'bbox': np.array([cx - bw/2, cy - bh/2, cx + bw/2, cy + bh/2], dtype=np.float32)
                                })
                    except Exception:
                        pass
                        
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        s = self.samples[idx]
        cv_img = cv2.imread(s['image_path'])
        if cv_img.shape[0] != 224 or cv_img.shape[1] != 224:
            cv_img = cv2.resize(cv_img, (224, 224))
        img_rgb = cv2.cvtColor(cv_img, cv2.COLOR_BGR2RGB)
        img_norm = (img_rgb.astype(np.float32) / 255.0 - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])
        img_tensor = torch.from_numpy(img_norm.transpose(2, 0, 1)).float()
        return img_tensor, s['class_id'], torch.from_numpy(s['bbox']).float()

dataset_dirs = glob.glob(os.path.join(Path.cwd(), "urban_dataset_*"))
if not dataset_dirs:
    dataset_dirs = glob.glob(os.path.join(parent_dir, "notebooks", "urban", "urban_dataset_*"))

bbox_dataset = UrbanBBoxDataset(dataset_dirs)
print(f"[*] Loaded Total {len(bbox_dataset)} Bounding Box Samples for 6 Urban Classes.")


### 2. Initialize MobileNetV2 Detector & Fine-Tuning Check

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pth_save_path  = os.path.join(Path.cwd(), "urban_detector.pth")
onnx_save_path = os.path.join(Path.cwd(), "urban_detector.onnx")

detector_model = MobileNetV2Detector(num_classes=len(DETECTION_CLASSES), pretrained=True).to(device)

if os.path.exists(pth_save_path):
    try:
        detector_model.load_state_dict(torch.load(pth_save_path, map_location=device))
        print(f"[+] Loaded existing Detector weights from '{pth_save_path}' for Fine-Tuning!")
    except Exception as e:
        print(f"[*] Initialized fresh MobileNetV2 detector weights.")


### 3. Interactive Detector Training UI (`ipywidgets`)

In [ ]:
import ipywidgets
from IPython.display import display

epochs_widget     = ipywidgets.IntText(description='epochs', value=10)
batch_size_widget = ipywidgets.IntText(description='batch size', value=8)
loss_widget       = ipywidgets.FloatText(description='loss')
progress_widget   = ipywidgets.FloatProgress(min=0.0, max=1.0, description='progress')
train_button      = ipywidgets.Button(description='Train & Export Detector ONNX', button_style='warning', icon='play')

sample_preview_widget = ipywidgets.Image(
    format='jpeg', width=224, height=224,
    layout=ipywidgets.Layout(border='2px solid #00ff00', border_radius='4px')
)

def export_detector_onnx():
    dummy_input = torch.randn(1, 3, 224, 224, device=device)
    try:
        torch.onnx.export(
            detector_model,
            dummy_input,
            onnx_save_path,
            verbose=False,
            input_names=['input_0'],
            output_names=['output_0'],
            opset_version=11,
            dynamo=False
        )
    except Exception:
        torch.onnx.export(
            detector_model,
            dummy_input,
            onnx_save_path,
            verbose=False,
            input_names=['input_0'],
            output_names=['output_0'],
            opset_version=11
        )
    print(f"[+] Exported Detector ONNX (Opset 11) -> '{onnx_save_path}'")

def start_detector_training(b):
    if len(bbox_dataset) == 0:
        print("[!] BBox Dataset is empty! Label objects with python -m jetracer.urban.label_bbox_gui first.")
        return
        
    epochs = epochs_widget.value
    batch_size = batch_size_widget.value
    train_loader = data.DataLoader(bbox_dataset, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(detector_model.parameters(), lr=1e-3)
    mse_loss = nn.MSELoss()
    ce_loss = nn.CrossEntropyLoss()
    
    train_button.disabled = True
    detector_model.train()
    start_t = time.time()
    
    print(f"\n[*] Starting Detector Training for {epochs} Epochs...")
    for epoch in range(epochs):
        processed = 0
        sum_loss = 0.0
        for images, cls_ids, bboxes in train_loader:
            images = images.to(device)
            cls_ids = cls_ids.to(device)
            bboxes = bboxes.to(device)
            
            optimizer.zero_grad()
            pred_bboxes, pred_cls = detector_model(images)
            
            loss_bbox = mse_loss(pred_bboxes, bboxes)
            loss_cls = ce_loss(pred_cls, cls_ids)
            loss = loss_bbox + loss_cls
            
            loss.backward()
            optimizer.step()
            
            count = len(cls_ids)
            processed += count
            sum_loss += float(loss) * count
            
            progress_widget.value = processed / len(bbox_dataset)
            loss_widget.value = sum_loss / processed
            
            # Update Live Image Preview
            try:
                img_np = images[0].cpu().numpy().transpose(1, 2, 0)
                img_np = (img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])) * 255.0
                img_np = np.clip(img_np, 0, 255).astype(np.uint8)
                img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
                
                box = pred_bboxes[0].detach().cpu().numpy()
                x1, y1, x2, y2 = int(box[0]*224), int(box[1]*224), int(box[2]*224), int(box[3]*224)
                cv2.rectangle(img_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2)
                
                sample_preview_widget.value = bgr8_to_jpeg(img_bgr)
            except Exception:
                pass
                
        print(f"  Epoch [{epoch+1:02d}/{epochs:02d}] - Loss: {loss_widget.value:.4f}")
        
    elapsed = time.time() - start_t
    print(f"[+] Detector Training finished in {elapsed:.1f}s!")
    
    detector_model.eval()
    torch.save(detector_model.state_dict(), pth_save_path)
    print(f"[+] Saved PyTorch Detector -> '{pth_save_path}'")
    export_detector_onnx()
    train_button.disabled = False

train_button.on_click(start_detector_training)

ui_layout = ipywidgets.VBox([
    sample_preview_widget,
    epochs_widget,
    batch_size_widget,
    progress_widget,
    loss_widget,
    train_button
])

display(ui_layout)
